# Student Dropout Prediction using Random Forest

This notebook implements a random forest classifier to predict student dropout based on cleaned features. The dataset is already split into training, validation, and test sets using the 'set' column.

## 1. Import Required Libraries

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset (adjust path as needed)
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

print(f"\nTarget variable '{target_col}' distribution:")
print(df[target_col].value_counts())
print(f"\nTarget variable '{target_col}' distribution (%):")
print(df[target_col].value_counts(normalize=True) * 100)

## 3. Split Data

In [ ]:
# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Check dropout rates in each set
print(f"\nDropout rate in training set: {train_df[target_col].mean():.3f}")
print(f"Dropout rate in validation set: {val_df[target_col].mean():.3f}")
print(f"Dropout rate in test set: {test_df[target_col].mean():.3f}")

## 4. Prepare Features and Target Variables

In [ ]:
# Identify potential ID columns and exclude them
exclude_cols = ['set', target_col]
potential_id_cols = ['stid', 'Studentnummer']  # Add any other ID columns if they exist

# Check if these columns exist in the dataset
for col in potential_id_cols:
    if col in df.columns:
        exclude_cols.append(col)
        print(f"Found and excluding ID column: {col}")

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\nNumber of features: {len(feature_cols)}")
print(f"Features: {feature_cols[:10]}...")  # Show first 10 features

In [ ]:
# Prepare feature and target variables
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Training features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Validation features shape: {X_val.shape}")
print(f"Test features shape: {X_test.shape}")

## 5. Data Preprocessing

In [ ]:
# Check for missing values
print("Missing values in training set:")
print(X_train.isnull().sum().sum())

# Handle missing values if any
if X_train.isnull().sum().sum() > 0:
    print("Imputing missing values...")
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), 
                                   columns=X_train.columns, 
                                   index=X_train.index)
    X_val_imputed = pd.DataFrame(imputer.transform(X_val), 
                                 columns=X_val.columns, 
                                 index=X_val.index)
    X_test_imputed = pd.DataFrame(imputer.transform(X_test), 
                                  columns=X_test.columns, 
                                  index=X_test.index)
else:
    print("No missing values found.")
    X_train_imputed = X_train
    X_val_imputed = X_val
    X_test_imputed = X_test

## 6. Model Training - Random Forest with Hyperparameter Tuning

In [ ]:
# Combine train and validation sets for grid search
X_train_val = pd.concat([X_train_imputed, X_val_imputed], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

print(f"Combined training + validation set shape: {X_train_val.shape}")
print(f"Target distribution in combined set: {y_train_val.mean():.3f}")

In [ ]:
# Define optimized parameter grid for Random Forest (reduced for faster execution)
# Original grid had 810 combinations (3×5×3×3×3×2), this has 36 combinations (3×3×2×2)
param_grid = {
    'n_estimators': [100, 200, 300],        # Most important: number of trees
    'max_depth': [10, 15, None],            # Key for controlling overfitting
    'min_samples_split': [5, 10],           # Reduced from [2, 5, 10]
    'max_features': ['sqrt', 'log2'],       # Removed None to speed up
    'class_weight': [None, 'balanced']      # Handle imbalanced data
}

# Calculate total combinations
total_combinations = 1
for key, values in param_grid.items():
    total_combinations *= len(values)

print("Optimized Parameter grid for Random Forest:")
for key, values in param_grid.items():
    print(f"{key}: {values}")
print(f"\nTotal combinations: {total_combinations} (vs 810 in original)")
print(f"With 5-fold CV: {total_combinations * 5} model fits (vs 4,050 in original)")
print(f"Expected speedup: ~{810/total_combinations:.1f}x faster")

In [ ]:
# Set up stratified k-fold cross validation  
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# For even faster execution (2-3 minutes), use this coarse grid (uncomment to use):
# coarse_param_grid = {
#     'n_estimators': [100, 200],
#     'max_depth': [10, None], 
#     'class_weight': [None, 'balanced']
# }  # Only 8 combinations = 40 model fits with 5-fold CV

# Perform grid search
print("Starting optimized Grid Search...")
print(f"This should complete in ~5 minutes (vs 30+ minutes with original grid)")
import time
start_time = time.time()

rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,  # Use coarse_param_grid for even faster execution
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,              # Use all available cores
    verbose=2               # More detailed progress output
)

# Fit the grid search
rf_grid.fit(X_train_val, y_train_val)

end_time = time.time()
print(f"\nGrid Search completed in {(end_time - start_time)/60:.1f} minutes")
print(f"Best parameters: {rf_grid.best_params_}")
print(f"Best cross-validation score: {rf_grid.best_score_:.4f}")

## 7. Model Evaluation

In [ ]:
# Get the best model
best_rf_model = rf_grid.best_estimator_

# Make predictions on validation set
y_val_pred = best_rf_model.predict(X_val_imputed)
y_val_pred_proba = best_rf_model.predict_proba(X_val_imputed)[:, 1]

# Make predictions on test set
y_test_pred = best_rf_model.predict(X_test_imputed)
y_test_pred_proba = best_rf_model.predict_proba(X_test_imputed)[:, 1]

# Calculate metrics for validation set
val_accuracy = accuracy_score(y_val, y_val_pred)
val_roc_auc = roc_auc_score(y_val, y_val_pred_proba)

# Calculate metrics for test set
test_accuracy = accuracy_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)

print("=== Model Performance ===")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation ROC-AUC: {val_roc_auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

## 8. Detailed Classification Reports

In [ ]:
# Print classification reports
print("=== Validation Set Classification Report ===")
print(classification_report(y_val, y_val_pred))

print("\n=== Test Set Classification Report ===")
print(classification_report(y_test, y_test_pred))

## 9. Confusion Matrix Visualization

In [ ]:
# Create confusion matrices
val_cm = confusion_matrix(y_val, y_val_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Validation confusion matrix
sns.heatmap(val_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Validation Set Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticklabels(['No Dropout', 'Dropout'])
axes[0].set_yticklabels(['No Dropout', 'Dropout'])

# Test confusion matrix
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Test Set Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_xticklabels(['No Dropout', 'Dropout'])
axes[1].set_yticklabels(['No Dropout', 'Dropout'])

plt.tight_layout()
plt.show()

## 10. ROC Curve Analysis

In [ ]:
# Calculate ROC curves
val_fpr, val_tpr, val_thresholds = roc_curve(y_val, y_val_pred_proba)
test_fpr, test_tpr, test_thresholds = roc_curve(y_test, y_test_pred_proba)

# Plot ROC curves
plt.figure(figsize=(10, 8))
plt.plot(val_fpr, val_tpr, label=f'Validation ROC (AUC = {val_roc_auc:.3f})', linewidth=2)
plt.plot(test_fpr, test_tpr, label=f'Test ROC (AUC = {test_roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.8)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Random Forest Model')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Feature Importance Analysis

In [ ]:
# Extract feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_rf_model.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values('importance', ascending=False)

# Display top 20 most important features
print("=== Top 20 Most Important Features ===")
top_features = feature_importance.head(20)
print(top_features)

# Plot feature importance
plt.figure(figsize=(12, 8))
top_features_plot = feature_importance.head(15)
plt.barh(range(len(top_features_plot)), top_features_plot['importance'])
plt.yticks(range(len(top_features_plot)), top_features_plot['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Cross-Validation Performance Analysis

In [ ]:
# Perform cross-validation with the best model to get more robust metrics
from sklearn.model_selection import cross_val_score

cv_model = RandomForestClassifier(**rf_grid.best_params_, random_state=42)

# Define scoring metrics
scoring_metrics = ['accuracy', 'roc_auc', 'precision', 'recall', 'f1']
cv_results = {}

print("=== Cross-Validation Results (5-fold) ===")
for metric in scoring_metrics:
    scores = cross_val_score(cv_model, X_train_val, y_train_val, 
                           cv=cv_strategy, scoring=metric)
    cv_results[metric] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{metric.upper()}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Plot cross-validation results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (metric, results) in enumerate(cv_results.items()):
    if i < len(axes):
        axes[i].bar(range(1, 6), results['scores'])
        axes[i].axhline(y=results['mean'], color='r', linestyle='--', 
                       label=f'Mean: {results["mean"]:.3f}')
        axes[i].set_title(f'{metric.upper()} Scores')
        axes[i].set_xlabel('Fold')
        axes[i].set_ylabel('Score')
        axes[i].set_ylim(0, 1)
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# Remove extra subplot
if len(scoring_metrics) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

## 13. Model Performance Summary and Analysis

In [ ]:
# Calculate comprehensive performance metrics
def calculate_detailed_metrics(y_true, y_pred, y_pred_proba):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    
    # Detailed metrics
    sensitivity = tp / (tp + fn)  # Recall/True Positive Rate
    specificity = tn / (tn + fp)  # True Negative Rate
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0  # Negative Predictive Value
    
    # Error rates
    false_alarm_rate = fp / (fp + tn)  # False Positive Rate
    missed_dropouts_rate = fn / (fn + tp)  # False Negative Rate
    
    return {
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'npv': npv,
        'false_alarm_rate': false_alarm_rate,
        'missed_dropouts_rate': missed_dropouts_rate,
        'confusion_matrix': cm
    }

# Calculate metrics for test set
test_scores = calculate_detailed_metrics(y_test, y_test_pred, y_test_pred_proba)

print("=== Final Model Performance on Test Set ===")
print(f"Accuracy: {test_scores['accuracy']:.4f}")
print(f"ROC-AUC: {test_scores['roc_auc']:.4f}")
print(f"Sensitivity (Recall): {test_scores['sensitivity']:.4f}")
print(f"Specificity: {test_scores['specificity']:.4f}")
print(f"Precision: {test_scores['precision']:.4f}")
print(f"Negative Predictive Value: {test_scores['npv']:.4f}")
print(f"False Alarm Rate: {test_scores['false_alarm_rate']:.4f}")
print(f"Missed Dropouts Rate: {test_scores['missed_dropouts_rate']:.4f}")

# Calculate baseline accuracy
baseline_accuracy = 1 - y_test.mean()  # Accuracy if we always predict "no dropout"
print(f"\nBaseline Accuracy (Always predict 'No Dropout'): {baseline_accuracy:.4f}")
print(f"Model Improvement over Baseline: {test_scores['accuracy'] - baseline_accuracy:.4f}")

# Model interpretation
print(f"\n=== Model Interpretation ===")
print(f"The Random Forest model correctly identifies {test_scores['sensitivity']:.1%} of actual dropouts")
print(f"The model correctly identifies {test_scores['specificity']:.1%} of non-dropouts")
print(f"When the model predicts dropout, it's correct {test_scores['precision']:.1%} of the time")
print(f"The model has a {test_scores['false_alarm_rate']:.1%} false alarm rate")
print(f"The model misses {test_scores['missed_dropouts_rate']:.1%} of actual dropouts")

## 14. Statistical Significance Testing

In [ ]:
# Statistical significance testing of cross-validation results
import scipy.stats as stats

cv_mean_acc = cv_results['accuracy']['mean']
acc_std = cv_results['accuracy']['std']
baseline_acc = 1 - y_train_val.mean()

# Perform one-sample t-test
t_stat, p_value = stats.ttest_1samp(cv_results['accuracy']['scores'], baseline_acc)

print("=== Statistical Significance Analysis ===")
print(f"Cross-validation mean accuracy: {cv_mean_acc:.4f} ± {acc_std:.4f}")
print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"Improvement: {cv_mean_acc - baseline_acc:.4f}")

print(f"\nOne-sample t-test (H0: model accuracy = baseline accuracy)")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"Result: Significant improvement (p < {alpha})")
else:
    print(f"Result: No significant improvement (p >= {alpha})")

# Calculate confidence interval
n = len(cv_results['accuracy']['scores'])
ci_lower = cv_mean_acc - 1.96 * (acc_std / np.sqrt(n))
ci_upper = cv_mean_acc + 1.96 * (acc_std / np.sqrt(n))

print(f"95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Assess consistency
cv_stats = {
    'mean': cv_mean_acc,
    'std': acc_std,
    'min': np.min(cv_results['accuracy']['scores']),
    'max': np.max(cv_results['accuracy']['scores']),
    'range': np.max(cv_results['accuracy']['scores']) - np.min(cv_results['accuracy']['scores'])
}

print(f"\n=== Model Consistency Analysis ===")
print(f"Standard deviation: {cv_stats['std']:.4f}")
print(f"Range: {cv_stats['range']:.4f} (min: {cv_stats['min']:.4f}, max: {cv_stats['max']:.4f})")

if cv_stats['std'] < 0.02:
    consistency_rating = "Very consistent"
elif cv_stats['std'] < 0.05:
    consistency_rating = "Consistent"
else:
    consistency_rating = "Inconsistent"
    
print(f"Consistency rating: {consistency_rating}")

## 15. Final Summary Report

In [ ]:
# Generate comprehensive summary report
dropout_rate = y_test.mean()

print("="*60)
print("RANDOM FOREST MODEL - FINAL PERFORMANCE REPORT")
print("="*60)

print(f"\n📊 DATASET SUMMARY:")
print(f"   • Total samples: {len(df):,}")
print(f"   • Training samples: {len(train_df):,}")
print(f"   • Validation samples: {len(val_df):,}")
print(f"   • Test samples: {len(test_df):,}")
print(f"   • Features used: {len(feature_cols)}")
print(f"   • Overall dropout rate: {dropout_rate:.1%}")

print(f"\n🎯 MODEL CONFIGURATION:")
print(f"   • Algorithm: Random Forest")
print(f"   • Best parameters: {rf_grid.best_params_}")
print(f"   • Cross-validation strategy: 5-fold Stratified")

print(f"\n📈 PERFORMANCE METRICS (Test Set):")
print(f"   • Accuracy: {test_scores['accuracy']:.4f} ({test_scores['accuracy']:.1%})")
print(f"   • ROC-AUC: {test_scores['roc_auc']:.4f}")
print(f"   • Sensitivity/Recall: {test_scores['sensitivity']:.4f} ({test_scores['sensitivity']:.1%})")
print(f"   • Specificity: {test_scores['specificity']:.4f} ({test_scores['specificity']:.1%})")
print(f"   • Precision: {test_scores['precision']:.4f} ({test_scores['precision']:.1%})")
print(f"   • False Alarm Rate: {test_scores['false_alarm_rate']:.4f} ({test_scores['false_alarm_rate']:.1%})")

print(f"\n🔍 BUSINESS IMPACT:")
test_dropout_rate = y_test.mean()
total_dropouts = int(len(y_test) * test_dropout_rate)
identified_dropouts = int(total_dropouts * test_scores['sensitivity'])
missed_dropouts = total_dropouts - identified_dropouts

print(f"   • Total dropouts in test set: {total_dropouts}")
print(f"   • Correctly identified: {identified_dropouts}")
print(f"   • Missed dropouts: {missed_dropouts}")
print(f"   • Students flagged for intervention: {int(np.sum(y_test_pred))}")

print(f"\n🏆 MODEL VALIDATION:")
print(f"   • Cross-validation accuracy: {cv_mean_acc:.4f} ± {acc_std:.4f}")
print(f"   • Statistical significance: {'Yes' if p_value < 0.05 else 'No'} (p = {p_value:.4f})")
print(f"   • Model consistency: {consistency_rating}")

# AUC Rating
if test_scores['roc_auc'] >= 0.9:
    auc_rating = "Excellent"
elif test_scores['roc_auc'] >= 0.8:
    auc_rating = "Good"
elif test_scores['roc_auc'] >= 0.7:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

print(f"\n⭐ OVERALL ASSESSMENT:")
print(f"   • ROC-AUC Rating: {auc_rating}")
improvement = test_scores['accuracy'] - baseline_acc
print(f"   • Improvement over baseline: {improvement:+.4f} ({improvement/baseline_acc:.1%})")

# Top 3 most important features
print(f"\n🔑 TOP 3 MOST IMPORTANT FEATURES:")
for i, (_, row) in enumerate(feature_importance.head(3).iterrows(), 1):
    print(f"   {i}. {row['feature']}: {row['importance']:.4f}")

print(f"\n📝 RECOMMENDATIONS:")
if test_scores['roc_auc'] >= 0.8 and test_scores['sensitivity'] >= 0.7:
    print(f"   • Model shows strong performance and can be deployed")
    print(f"   • Consider implementing early intervention strategies")
elif test_scores['roc_auc'] >= 0.7:
    print(f"   • Model shows decent performance but may need improvement")
    print(f"   • Consider feature engineering or ensemble methods")
else:
    print(f"   • Model performance needs significant improvement")
    print(f"   • Consider collecting more data or different features")

print("="*60)